In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 샘플링 주파수를 200으로 설정(1초에 데이터 200개 생성)
# 나이퀴스트 주파수 기준으로는 최대 100Hz(200 / 2)까지 주파수 성분을 왜곡 없이 분석 가능
fs = 200
# 0~0.9999 초까지 1/200(0.005) 단위로 쪼갠다.
t = np.arange(0, 1, 1 / fs)

# 5Hz 와 20Hz 가 섞인 신호를 만든다.
x = np.sin(2 * np.pi * 5 * t) + (0.5 * np.sin(2 * np.pi * 20 * t))   # 5Hz + 20Hz


In [ ]:
# rfftfreq(샘플갯수,샘플링간격) : 길이와, 간격을 넣으면 분석할 Hz를 만들어준다.
# Hz 자체가 1초에 얼마나 진동하느냐이므로 길이와 간격에 따라 폭이 달라질수 밖에 없다. 
freq = np.fft.rfftfreq(len(x), 1 / fs)
# freq # [0,...,100]

# 신호 x를 주파수별 실제 파형 크기(진폭)로 바꿔 준다.
r1 = np.fft.rfft(x) # x 안에 섞여 있는 5Hz, 20Hz 주파수 추출(복소수형태)
# print(r1) # -1.09005945e-14+0.00000000e+00j
r2 = np.abs(r1) # 허수부를 버리고 절대값으로 변경하여 순수 크기만 남김
# 살펴보다 보면 이상점이 있다.
# [5]1.00000000e+02 -> 100.0, [20]5.00000000e+01 -> 50.0 나머지는 거의 0에 가까움
# print(r2)

In [23]:
# 이유는 5Hz 와 50Hz 가 발견된 것이고, 나머지 주파수는 잡힌 데이터가 없어서 이다.
# 그런데 우리는 5Hz : np.sin(), 20Hz : 0.5 * np.sin(2 * np.pi * 20 * t) 을 사용하였기에 1,0.5가 나와야 한다.
# 그런데 100과 50은 너무 크다. 그 이유는 데이터 수 때문이다.
# 데이터 내 각 Hz가 발견되면 누적합을 하는데, 이러면 데이터가 많아질수록 왜곡이 더 커진다.
# 그리하여 갯수만큼 나눠 준다.
r3 = r2/len(x)
print(f'[5] : {r3[5]:.3f} [20] : {r3[20]:.3f}')
# 주파수는 0을 기준으로 위아래로 파종을 치는데, np.fft.rfft(x) 는 양의 주파수만 가져온다.
# 그래서 2를 곱해져야 제대로된 진종폭(우리의 예시 상으로는 0.5와 1)
amp = r3 * 2
print(f'[5] : {amp[5]:.3f} [20] : {amp[20]:.3f}')
# amp # [1.09005945e-16,...,1.53390340e-17]

[5] : 0.500 [20] : 0.250
[5] : 1.000 [20] : 0.500


In [ ]:
# row,col,size
fig, ax = plt.subplots(1, 2, figsize=(11, 3))
# line chart
ax[0].plot(t, x) 
ax[0].set_title('time domain')
# 
ax[1].stem(freq, amp) # bar 는 너무 두꺼워서 주파수 처럼 촘촘한 대역을 처리하기가 어렵다.
ax[1].set_title('frequency domain'); 
ax[1].set_xlim(0, 40)
plt.show()      # → 5Hz(크기 1), 20Hz(크기 0.5) 두 막대가 보인다